In [2]:
import sqlite3
import pathlib
import os
from dataclasses import astuple
import pandas as pd
from equity_ops import storage
from equity_ops.contracts import TradeEvent
import logging
logging.basicConfig(level=logging.INFO)


In [ ]:
conn = sqlite3.connect(storage.equity_ops_lab / "db" / "trades.db")
# event = TradeEvent("T2", 1, "ABC", 100, 1000, False)
# conn.execute("""
#             INSERT INTO trade_events(trade_id,version,symbol,signed_quantity,price_in_pence,cancelled)
#             VALUES(?,?,?,?,?,?)
#         """, (*astuple(event),))
# conn.commit()
storage.initialise_db(conn)
for event in = TradeEvent("T2", 1, "ABC", 100, 1000, False)
try:
    if storage.store_event(conn, event):
        logging.info(f"Stored trade_id {event.trade_id}")
    else:
        logging.info(f"Duplicate DB entry for trade_id {event.trade_id}")
except ValueError as v:
    logging.info(str(v))

In [12]:
cur = conn.execute("""
CREATE TABLE IF NOT EXISTS trade_events (
    trade_id TEXT NOT NULL,
    version INTEGER NOT NULL,
    symbol TEXT NOT NULL,
    signed_quantity INTEGER NOT NULL,
    price_in_pence INTEGER NOT NULL,
    cancelled INTEGER NOT NULL CHECK (cancelled IN (0, 1)),
    PRIMARY KEY (trade_id, version)
);
    """)

In [13]:
conn.commit()

In [ ]:
from typing import Iterable
import logging

def positions_and_cash(trades: Iterable[tuple[str, int, int]]) -> dict[str, tuple[int, int]]:
    r = {}
    for trade in trades:
        symbol, signed_quantity, price_in_pence = trade
        q, c = r.get(symbol, (0, 0))
        q += signed_quantity
        c += -signed_quantity*price_in_pence
        logging.debug(f"{symbol=}, {signed_quantity=}, {price_in_pence=}, {q=}; {c=}")
        r[symbol] = (q, c)
    return r

In [ ]:
active

In [ ]:
equity_ops_lab = pathlib.Path(os.getcwd()).resolve()
logging.info(equity_ops_lab)
conn = sqlite3.connect(equity_ops_lab / "db" / "trades.db")
df = pd.read_sql_query("SELECT * FROM trade_events", conn)

active = df[df["cancelled"] == 0]
active["cash_in_pence"] = -active["signed_quantity"]*active["price_in_pence"]
positions = active.groupby("symbol")[["signed_quantity", "cash_in_pence"]].sum()


INFO:root:/home/richard/rthickling/equity-ops-lab


In [8]:
positions

,signed_quantity,cash_in_pence
symbol,,
ABC,280,-269225
DEF,124,-149618
FLAT,12,-11810
GHI,2,-20002
LMN,270,-93125
OLD,11,-44000
QRS,85,-678892
VOID,30,-14000
XYZ,-100,257375


In [9]:
summary = positions.reset_index()
summary


,symbol,signed_quantity,cash_in_pence
0,ABC,280,-269225
1,DEF,124,-149618
2,FLAT,12,-11810
3,GHI,2,-20002
4,LMN,270,-93125
5,OLD,11,-44000
6,QRS,85,-678892
7,VOID,30,-14000
8,XYZ,-100,257375


In [11]:
summary.rename(columns={"signed_quantity": "position"})

,symbol,position,cash_in_pence
0,ABC,280,-269225
1,DEF,124,-149618
2,FLAT,12,-11810
3,GHI,2,-20002
4,LMN,270,-93125
5,OLD,11,-44000
6,QRS,85,-678892
7,VOID,30,-14000
8,XYZ,-100,257375


In [24]:

active["cash_in_pence"] = -active["signed_quantity"]*active["price_in_pence"]

In [25]:
active

,trade_id,version,symbol,signed_quantity,price_in_pence,cancelled,cash_in_pence
0,T1,1,ABC,100,1000,0,-100000
1,T2,1,ABC,-40,1200,0,48000
2,T1,2,ABC,120,1000,0,-120000
